# Step 7: Trials Count Analysis

# Mounting Google Drive

In [ ]:
from google.colab import drive
drive.mount('/gdrive')
%cd /gdrive

Mounted at /gdrive
/gdrive


# Move to the Dataset Directory in My Drive

In [ ]:
import os
os.chdir("/gdrive/My Drive/Machine Learning (HPL) - MITACS 2025")
!pwd

/gdrive/My Drive/Machine Learning (HPL) - MITACS 2025


# Trials Count Analysis Code:

In [4]:
"""
Created on Fri Jul 19 17:31:35 2024

@author: roqui
"""

import pandas as pd

def caller(feat, class_model, model_name, model, df_stats, speed, df, number_class):

    temp_taking_class_type     = model[(model == feat).any(axis=1) & (model == class_model).any(axis=1)]
    temp_taking_class_type_all = model[(model == class_model).any(axis=1)]

    # Merge and find rows in temp_taking_class_type_all that are not in temp_taking_class_type
    temp_taking_class_type_remaining = temp_taking_class_type_all.merge(temp_taking_class_type, how='left', indicator=True)
    temp_taking_class_type_remaining = temp_taking_class_type_remaining[temp_taking_class_type_remaining['_merge'] == 'left_only'].drop(columns=['_merge'])

    # Perform the merge
    result = df.loc[temp_taking_class_type.index]
    result = result.reset_index()
    result = pd.concat([temp_taking_class_type.reset_index(), result.iloc[:,5:]], axis=1)

    # also get the remaining trials data
    result_remaining = df.loc[temp_taking_class_type_remaining.index]
    result_remaining = result_remaining.reset_index()
    result_remaining = pd.concat([temp_taking_class_type_remaining.reset_index(), result_remaining.iloc[:,5:]], axis=1)

    higher_than_mean_counter, higher_than_mean_counter_remaining = 0, 0
    lower_than_mean_counter, lower_than_mean_counter_remaining = 0, 0
    within_std_counter = 0

    Original_Data_Mean = df_stats[(df_stats.iloc[:,0] == feat) & (df_stats["Group"] == class_model)]["Mean"].values[0]
    Original_Data_Std  = df_stats[(df_stats.iloc[:,0]== feat) & (df_stats["Group"] == class_model)]["Std Deviation"].values[0]

    for i in range(len(result)):
        if result.loc[i, feat] >= Original_Data_Mean:
            higher_than_mean_counter += 1
        else:
            lower_than_mean_counter += 1
        if (result.loc[i, feat] > Original_Data_Mean - Original_Data_Std) or (result.loc[i, feat] < Original_Data_Mean + Original_Data_Std):
            within_std_counter += 1

    for i in range(len(result_remaining)):
        if result_remaining.loc[i, feat] >= Original_Data_Mean:
            higher_than_mean_counter_remaining += 1
        else:
            lower_than_mean_counter_remaining += 1

    # print("higher_than_mean_counter = ", higher_than_mean_counter)
    # print("lower_than_mean_counter = ", lower_than_mean_counter)
    # print("within_std_counter = ", within_std_counter)

    Number_trials_ft  = None
    Number_trials_top = len(temp_taking_class_type)
    Trials_higher     = higher_than_mean_counter/len(temp_taking_class_type)*100
    Trials_lower      = lower_than_mean_counter/len(temp_taking_class_type)*100
    Remaining_higher  = higher_than_mean_counter_remaining/(number_class-len(temp_taking_class_type))*100
    Remaining_lower   = lower_than_mean_counter_remaining/(number_class-len(temp_taking_class_type))*100
    wrong_value       = (min(higher_than_mean_counter, lower_than_mean_counter) + min(higher_than_mean_counter_remaining, lower_than_mean_counter_remaining))/4
    wrong_sum         = None
    Data_mean         = "({}, {})".format(higher_than_mean_counter, lower_than_mean_counter)
    Remaing_trials    = "({}, {})".format(higher_than_mean_counter_remaining, lower_than_mean_counter_remaining)


    return [model_name, class_model, feat, Number_trials_ft, Number_trials_top, Trials_higher, Trials_lower, Remaining_higher, Remaining_lower,
            wrong_value, wrong_sum, Data_mean, Remaing_trials, Original_Data_Mean, Original_Data_Std]

    kkk = {"Model" : model_name,
           "Class" : class_model,
           "Top_1_Feature" : feat,
           "# of Trials Y+O for each feature in Top-3 rank (out of 400)" : None,
           "# of Trials with feature in Top-3 rank (out of 200)" : "{}".format(len(temp_taking_class_type)), # divide by 2 because there are 200 Y and 200 O.
           # "Trials with data (higher, lower) than mean" : "({}, {})".format(higher_than_mean_counter, lower_than_mean_counter),
           "Higher (%)" : "={}/{} * 100".format(higher_than_mean_counter, len(temp_taking_class_type)),
           "Lower (%)" : "={}/{} * 100".format(lower_than_mean_counter, len(temp_taking_class_type)),
           # "Remaining trials (higher, lower) than mean" : "({}, {})".format(higher_than_mean_counter_remaining, lower_than_mean_counter_remaining),
           "Remaining Higher (%)" : "={}/{} * 100".format(higher_than_mean_counter_remaining, 200-len(temp_taking_class_type)),
           "Remaining Lower (%)" : "={}/{} * 100".format(lower_than_mean_counter_remaining, 200-len(temp_taking_class_type)),
           "Wrong (%)" : "=({}+{}) / 4".format(min(higher_than_mean_counter, lower_than_mean_counter), min(higher_than_mean_counter_remaining, lower_than_mean_counter_remaining)),

           "Top-3 Trials with data (higher, lower) than mean" : "({}, {})".format(higher_than_mean_counter, lower_than_mean_counter),
           "Remaining trials (higher, lower) than mean" : "({}, {})".format(higher_than_mean_counter_remaining, lower_than_mean_counter_remaining),

           "Original_Data_Mean" : Original_Data_Mean,
           "Original_Data_Std" : Original_Data_Std,
           }

    print(kkk)

def crate_df_result(number_of_class):
    columns = pd.MultiIndex.from_tuples([
        ('', 'Model'),
        ('', 'Class'),
        ('', 'Feature'),
        ('', f'# of Trials Y+O for each feature in top 3 rank (Out of {number_of_class[0]+number_of_class[1]})'),
        ('', f'# Trials with top feature in top 3 ranking (Y:{number_of_class[0]} O:{number_of_class[1]})'),
        ('Trials with data (higher, lower) than mean', 'Higher (%)'),
        ('Trials with data (higher, lower) than mean', 'Lower (%)'),
        ('Remaining trials (higher, lower) than mean', 'Higher (%)'),
        ('Remaining trials (higher, lower) than mean', 'Lower (%)'),
        ('', '% Wrong'),
        ('', 'Sum of Wrong (%)'),
        ('', 'Trials with data (higher, lower) than mean'),
        ('', 'Remaining trials (higher, lower) than mean'),
        ('', 'Original_Data_Mean'),
        ('', 'Original_Data_Std')
    ])

    df_result = pd.DataFrame(columns=columns)

    return df_result

# Folder
path_folder = 'Sample Data'
path_train  = f'{path_folder}/ALL FEATURES OLDER - Pes Planus and Control JUNE 25 Train.xlsx'
path_test   = f'{path_folder}/ALL FEATURES OLDER - Pes Planus and Control JUNE 25 Test.xlsx'
path_stats  = f'{path_folder}/Feature_stats_older.xlsx'
path_shap   = f'{path_folder}/TOPs_SHAP_DATA.xlsx'
path_models = f'{path_folder}/Features_SHAP_TOP.xlsx'

all_class_models = ["Control", "Flatfoot"]

# Read the Excel file
EXCEL_train = pd.ExcelFile(path_train)
EXCEL_test = pd.ExcelFile(path_test)
EXCEL_stats = pd.ExcelFile(path_stats)
EXCEL_SHAP = pd.ExcelFile(path_shap)

df_main  = {}
df_stats = {}
df_models = {}
df_features = {}

ft_models = pd.read_excel(path_models, index_col=0)
models    = ft_models.keys()

for model in models:
    model_temp = model.split('_')
    df_models[model_temp[0]] = {}
    df_features[model_temp[0]] = {}
for model in models:
    features = ft_models[model].dropna().to_numpy()
    model_temp = model.split('_')
    df_shap = pd.read_excel(EXCEL_SHAP, sheet_name=model, index_col = 0)
    df_models[model_temp[0]][model_temp[1]] = df_shap
    df_features[model_temp[0]] [model_temp[1]] = features

for sheet_name in EXCEL_train.sheet_names:
    df_train = pd.read_excel(EXCEL_train, sheet_name=sheet_name)
    df_test  = pd.read_excel(EXCEL_test, sheet_name=sheet_name)
    df_stats_ex = pd.read_excel(EXCEL_stats, sheet_name=sheet_name)

    # merge train and test sets
    df_main[sheet_name] = pd.concat([df_train, df_test], axis=0)
    df_main[sheet_name].reset_index(drop=True,inplace=True)
    # df stats
    df_stats[sheet_name] = df_stats_ex

df_results   = {}
df_data_set  = {}
for speed in df_models:
    df_results[speed] = {}
    df_data_set[speed]  = {}
    number_of_class   =  df_main[speed]['Group'].value_counts()
    for model in df_models[speed]:
        df_models_t = df_models[speed][model]
        df_data_set[speed][model] = pd.concat([df_models_t.iloc[:,0:4], df_models_t.iloc[:,-6:-1]], axis = 1)
        df_results[speed][model] = crate_df_result(number_of_class)


for speed in df_models:
    counter = 0
    number_of_class  =  df_main[speed]['Group'].value_counts()
    for model in df_models[speed]:
        for feat in df_features[speed][model]:
            for class_model in all_class_models:
                df_results[speed][model].loc[counter] = caller(feat, class_model, model, df_data_set[speed][model], df_stats[speed], speed, df_main[speed], number_of_class[class_model])
                counter += 1
        counter = 0

# Save the updated DataFrames back to an Excel file with multiple sheets
with pd.ExcelWriter(f'{path_folder}/Trials_count_model.xlsx') as writer:
    for speed in df_results:
        for model in df_results[speed]:
            df_results[speed][model].to_excel(writer, sheet_name=f'{speed}_{model}')

<ipython-input-4-2091426778>:95: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ('', f'# of Trials Y+O for each feature in top 3 rank (Out of {number_of_class[0]+number_of_class[1]})'),
<ipython-input-4-2091426778>:96: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ('', f'# Trials with top feature in top 3 ranking (Y:{number_of_class[0]} O:{number_of_class[1]})'),
<ipython-input-4-2091426778>:95: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[